# ChlamyCodonTransformer Fine-tuning

C. reinhardtii 葉緑体コドン最適化のための CodonTransformer ファインチューニング

**実行前チェックリスト:**
1. ランタイム → ランタイムのタイプを変更 → **GPU (A100推奨, T4でも可)**
2. `training_data.json` を Google Drive の `MyDrive/ChlamyDesign/data/` にアップロード済み
3. （任意）HuggingFace token を取得 (write権限): https://huggingface.co/settings/tokens

## 0. Google Drive マウント & パス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR     = '/content/drive/MyDrive/ChlamyDesign'
TRAINING_DATA = f'{DRIVE_DIR}/data/training_data.json'
MODEL_OUTPUT  = f'{DRIVE_DIR}/models/ChlamyCodonTransformer'
LOG_DIR       = f'{DRIVE_DIR}/logs'

os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

assert os.path.exists(TRAINING_DATA), (
    f'training_data.json が見つかりません:\n  {TRAINING_DATA}\n'
    f'Google Drive の MyDrive/ChlamyDesign/data/ にアップロードしてください。'
)

import json
with open(TRAINING_DATA) as f:
    lines = f.readlines()
N_SAMPLES = len(lines)
print(f'✓ 訓練データ: {TRAINING_DATA}')
print(f'✓ サンプル数: {N_SAMPLES:,}')
print(f'✓ モデル出力: {MODEL_OUTPUT}')

## 1. パッケージインストール & GPU確認

In [ ]:
!pip install -q 'CodonTransformer>=1.0.0' 'transformers>=4.30.0' 'accelerate>=0.20.0'

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA使用可 : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
else:
    print('⚠ GPU が検出されません。ランタイムの設定を確認してください。')

## 2. ハイパーパラメータ設定

In [ ]:
import math

# ── ハイパーパラメータ (CLAUDE.md Phase 3 仕様) ──
BATCH_SIZE   = 6
MAX_EPOCHS   = 10
LR           = 5e-5
WARMUP_RATIO = 0.1

# IterableDataset は num_train_epochs が使えないため max_steps で指定
STEPS_PER_EPOCH = math.ceil(N_SAMPLES / BATCH_SIZE)
MAX_STEPS       = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS    = int(MAX_STEPS * WARMUP_RATIO)

print(f'サンプル数      : {N_SAMPLES}')
print(f'バッチサイズ    : {BATCH_SIZE}')
print(f'1エポックのstep : {STEPS_PER_EPOCH}')
print(f'合計 steps      : {MAX_STEPS}  ({MAX_EPOCHS} epochs)')
print(f'warmup steps    : {WARMUP_STEPS}')

## 3. モデル & トークナイザー読み込み

In [ ]:
from transformers import BigBirdForMaskedLM
from CodonTransformer.CodonPrediction import load_tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {device}')

print('ベースモデルを HuggingFace Hub から読み込み中...')
model     = BigBirdForMaskedLM.from_pretrained('adibvafa/CodonTransformer')
tokenizer = load_tokenizer()

n_params = sum(p.numel() for p in model.parameters())
print(f'パラメータ数: {n_params:,}')
print('✓ 読み込み完了')

## 4. データセット & コレーター準備

In [ ]:
from CodonTransformer.CodonUtils import IterableJSONData
from transformers import DataCollatorForLanguageModeling

train_dataset = IterableJSONData(
    data_path=TRAINING_DATA,
    train=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,  # 標準MLM: 15%マスク
)

print('✓ データセット準備完了')

## 5. HuggingFace Hub ログイン（任意）

In [ ]:
# HuggingFace にアップロードする場合のみ設定
PUSH_TO_HUB  = False               # True にするとアップロード
HF_USERNAME  = 'your_hf_username'  # ← 自分のユーザー名に変更
HF_REPO_ID   = f'{HF_USERNAME}/ChlamyCodonTransformer'

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # トークン入力プロンプトが表示されます
    print(f'✓ HuggingFace Hub ログイン完了: {HF_REPO_ID}')
else:
    print('HuggingFace アップロード: 無効 (PUSH_TO_HUB=True で有効化)')

## 6. ファインチューニング実行

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    # IterableDataset は max_steps で制御
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    # ログ & 保存
    logging_dir=LOG_DIR,
    logging_steps=50,
    save_steps=STEPS_PER_EPOCH,  # 1エポックごとに保存
    save_total_limit=3,
    # HuggingFace Hub
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HF_REPO_ID if PUSH_TO_HUB else None,
    # 高速化
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f'ファインチューニング開始')
print(f'  lr={LR}, batch={BATCH_SIZE}, steps={MAX_STEPS}, warmup={WARMUP_STEPS}')
trainer.train()

## 7. モデル保存

In [ ]:
# HuggingFace 形式で保存 (Google Drive)
final_path = f'{MODEL_OUTPUT}/final'
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f'✓ HF形式で保存: {final_path}')

# .pt 形式でも保存 (src/model/predict.py 互換)
pt_path = f'{MODEL_OUTPUT}/ChlamyCodonTransformer.pt'
torch.save(model.state_dict(), pt_path)
print(f'✓ .pt で保存   : {pt_path}')

if PUSH_TO_HUB:
    trainer.push_to_hub(commit_message='ChlamyCodonTransformer: fine-tuned on C.reinhardtii chloroplast CDS')
    print(f'✓ HuggingFace Hub: https://huggingface.co/{HF_REPO_ID}')

## 8. 動作確認 — rbcL タンパク質でテスト

In [ ]:
from CodonTransformer.CodonPrediction import predict_dna_sequence

# rbcL (C. reinhardtii 葉緑体 RuBisCO 大サブユニット)
test_protein = 'MSPQTETKASVGFKAGVKEYKLTYYTPEYETKDTDILAAFRVTPQPGVPPEEAGAAVAAESSTGTWTTVWTDGLTSLDRYKGRCYRIEEGQTINEDTAYFFDQFKQGCDINIFQYDSAGLLNKFEEMDHYREPEHSPESAGGIHVWHMPALTEIFGDDSVLQFGGGTLGHPWGNAPGATN_'

model.eval()
result = predict_dna_sequence(
    protein=test_protein,
    organism='Chlamydomonas reinhardtii chloroplast',
    device=device,
    model=model,
    tokenizer=tokenizer,
)

dna = result.predicted_dna
gc  = sum(1 for b in dna if b in 'GC') / len(dna) * 100

print(f'予測DNA (先頭90 nt): {dna[:90]}')
print(f'GC%: {gc:.1f}%  (目標: ~34%, ベースCT: ~61%)')
print(f'長さ: {len(dna)} nt')

# ベースモデルとの比較のために再読み込みして比較
base_model = BigBirdForMaskedLM.from_pretrained('adibvafa/CodonTransformer').to(device)
base_result = predict_dna_sequence(
    protein=test_protein,
    organism='Chlamydomonas reinhardtii chloroplast',
    device=device, model=base_model, tokenizer=tokenizer,
)
base_gc = sum(1 for b in base_result.predicted_dna if b in 'GC') / len(base_result.predicted_dna) * 100

print(f'\n--- 比較 ---')
print(f'ChlamyCT  GC%: {gc:.1f}%')
print(f'Base CT   GC%: {base_gc:.1f}%')
print(f'改善量       : {base_gc - gc:+.1f}pp')